In [ ]:
# Canonical analysis definitions (single source of truth: analysis/_canonical_defs.py).
# Importing these overrides any local fallback definitions later in the notebook.
# Edits to filters / score-mapping / bucket-rule belong in _canonical_defs.py, not here.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("analysis").resolve()))
from _canonical_defs import (
    ANCHOR, L2_CELLS, AXIS_CELLS, VERDICT_SCORE,
    candidate_score, sign,
    any_tier_disagreement, boundary_flip, strict_reversal,
    same_side_magnitude_shift, strong_to_weak_shift,
    assign_bucket,
)


# 10c — Axis fingerprints

*Renamed from `06c_axis_fingerprints.ipynb` 2026-05-25.*: what kind of prompts does each axis flip?

Closes the "do the axes do *different work*?" question. For each framework axis (C, F, D, L1) and each volatility regime (stable_core / maximally_unstable / axis-only), characterise:

1. **Style fingerprint** — do flipped prompts have larger length / markdown differentials than non-flipped?
2. **Margin fingerprint** — do flips concentrate on prompts where the anchor verdict was already near a tie?
3. **Prompt-feature fingerprint** — prompt length, has-code-block, has-numbers per axis.
4. **Volatility-regime fingerprint** — six regimes with numerical signatures (anchor margin, language, code-density, length).
5. **Regex feature corroboration** — truncation (dropped — no signal) + multi-part (kept as descriptive only).
6. **Hand-picked example prompts** — 5 per regime for §5/appendix qualitative texture.
7. **Headline summary table** — per-axis + per-regime fingerprints in one table.

LLM-assisted regime descriptions and per-prompt tags (in earlier drafts) tried and dropped 2026-05-04 — confabulation risk + reviewer optics. See methodology_notes.md "06c volatility-regime fingerprints and LLM-tagging audit" subsection.

**Inputs:** existing CSVs from nb 06 (no new judge calls).

**Outputs:**
- `results/axis_x_style.csv` — flipped vs non-flipped style-feature distributions per axis
- `results/axis_x_margin.csv` — flipped vs non-flipped anchor-margin distributions per axis
- `results/axis_x_prompt_features.csv` — prompt-level features per axis
- `results/regime_fingerprints.csv` — per-regime numerical fingerprints + regex features
- `results/regime_summaries_llm.json` — audit trail (LLM-tagging dropped; kept for record)

**Cost:** all cells free (no API calls).

In [1]:
# [L1_GEMINI_PATCH] This cell conditionally handles L1_claude. After Gemini
# data lands in verdict_scores_wide.csv, the same logic applies to L1_gemini.
# Manual extension may be needed depending on what this cell computes — review
# the 'L1_claude' references below and parallel them for 'L1_gemini'.
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu

results_dir = Path('results')

wide = pd.read_csv(results_dir / 'verdict_scores_wide.csv', index_col='uid')
style_df = pd.read_csv(results_dir / 'style_features.csv')
volatility = pd.read_csv(results_dir / 'per_prompt_volatility.csv')
pairs = pd.read_json('data/preprocessed_pairs.jsonl', lines=True)

ANCHOR = 'C1F1D1'
AXIS_CELLS = {'C': 'C2F1D1', 'F': 'C1F2D1', 'D': 'C1F1D2', 'L1': 'L1_claude'}
if 'L1_gemini' in wide.columns:  # [L1_GEMINI_PATCH]
    AXIS_CELLS['L1_gemini'] = 'L1_gemini'

# Anchor-margin: |mean_score| at anchor — proxy for how close to a tie the verdict was
anchor_margin = wide[ANCHOR].abs().rename('anchor_margin')

# Per-axis flip sets (uid → bool) vs anchor
def axis_flip_mask(wide, axis_cell, anchor=ANCHOR):
    a = wide[anchor]
    b = wide[axis_cell]
    common = a.notna() & b.notna()
    return common & (np.sign(a) != np.sign(b))

axis_masks = {ax: axis_flip_mask(wide, cell) for ax, cell in AXIS_CELLS.items()}

print(f'Loaded: {wide.shape[0]} prompts × {wide.shape[1]} cells')
print(f'Style features: {style_df.shape[0]} rows')
print(f'Volatility profile: {volatility.shape[0]} rows')
print('\nAxis flip-set sizes vs anchor:')
for ax, mask in axis_masks.items():
    print(f'  {ax}: {mask.sum()} prompts')

Loaded: 500 prompts × 10 cells
Style features: 500 rows
Volatility profile: 496 rows

Axis flip-set sizes vs anchor:
  C: 129 prompts
  F: 208 prompts
  D: 110 prompts
  L1: 173 prompts
  L1_gemini: 184 prompts


## 1. Axis × style features

For each axis, compare the distribution of style differentials (`length_diff`, `md_diff`) between flipped and non-flipped prompts. Mann-Whitney p-value tests whether style differs significantly between the two sets.

**Interpretation:** if `length_diff` mean is HIGHER for flipped prompts than non-flipped, that axis preferentially flips prompts where the candidate is much longer than the baseline — i.e., the axis is partly style-mediated, not pure rubric/format/scope work.

In [2]:
style_indexed = style_df.set_index('uid')

rows = []
for ax, mask in axis_masks.items():
    flipped_uids = mask[mask].index
    nonflipped_uids = mask[~mask].index
    flipped = style_indexed.loc[style_indexed.index.intersection(flipped_uids)]
    nonflipped = style_indexed.loc[style_indexed.index.intersection(nonflipped_uids)]
    for feat in ['length_diff', 'md_diff']:
        f_vals = flipped[feat].dropna()
        n_vals = nonflipped[feat].dropna()
        try:
            _, p = mannwhitneyu(f_vals, n_vals, alternative='two-sided')
        except ValueError:
            p = np.nan
        rows.append({
            'axis': ax,
            'feature': feat,
            'flipped_n': len(f_vals),
            'flipped_mean': f_vals.mean(),
            'flipped_median': f_vals.median(),
            'nonflipped_n': len(n_vals),
            'nonflipped_mean': n_vals.mean(),
            'nonflipped_median': n_vals.median(),
            'mean_diff': f_vals.mean() - n_vals.mean(),
            'mw_pvalue': p,
        })

axis_x_style = pd.DataFrame(rows)
print('Axis × style features (flipped vs non-flipped):')
print(axis_x_style.round(4).to_string(index=False))

axis_x_style.to_csv(results_dir / 'axis_x_style.csv', index=False)
print('\nSaved: results/axis_x_style.csv')

print('\n--- Headline reading ---')
for ax in AXIS_CELLS:
    sub = axis_x_style[axis_x_style.axis == ax]
    for _, row in sub.iterrows():
        sig = '**' if row.mw_pvalue < 0.05 else '  '
        direction = 'longer/more' if row.mean_diff > 0 else 'shorter/less'
        print(f'  {ax} × {row.feature}: flipped prompts have {direction} (Δmean={row.mean_diff:+.4f}, p={row.mw_pvalue:.3f}) {sig}')

Axis × style features (flipped vs non-flipped):
     axis     feature  flipped_n  flipped_mean  flipped_median  nonflipped_n  nonflipped_mean  nonflipped_median  mean_diff  mw_pvalue
        C length_diff        129        0.1035          0.0563           371           0.0991             0.0783     0.0045     0.6868
        C     md_diff        129        0.0190          0.0174           371           0.0188             0.0154     0.0002     0.5061
        F length_diff        208        0.1199          0.0804           292           0.0862             0.0696     0.0336     0.1946
        F     md_diff        208        0.0191          0.0174           292           0.0186             0.0148     0.0006     0.3414
        D length_diff        110        0.1100          0.0536           390           0.0975             0.0804     0.0125     0.8498
        D     md_diff        110        0.0211          0.0181           390           0.0182             0.0150     0.0030     0.0403
       

## 2. Axis × verdict-margin

For each axis, compare anchor-margin (`|mean_score|` at C1F1D1) between flipped and non-flipped prompts. Low anchor-margin = the original verdict was close to a tie (already borderline). High anchor-margin = the original verdict was decisive.

**Interpretation:** if flipped prompts have systematically LOWER anchor-margin than non-flipped, the axis only tips already-borderline prompts (consistent with "any axis tips the borderline subset"). If flipped prompts span a wide margin range, the axis can move decisive prompts too (genuinely independent work).

In [3]:
rows = []
for ax, mask in axis_masks.items():
    common = mask.dropna().index
    flipped = anchor_margin.loc[common[mask.loc[common]]]
    nonflipped = anchor_margin.loc[common[~mask.loc[common]]]
    try:
        _, p = mannwhitneyu(
            flipped.dropna(),
            nonflipped.dropna(),
            alternative='two-sided',
            method='asymptotic',
        )
    except (ValueError, Exception):
        p = np.nan
    rows.append({
        'axis': ax,
        'flipped_n': len(flipped),
        'flipped_margin_mean': flipped.mean(),
        'flipped_margin_median': flipped.median(),
        'flipped_pct_decisive': 100 * (flipped >= 1.0).mean(),
        'nonflipped_n': len(nonflipped),
        'nonflipped_margin_mean': nonflipped.mean(),
        'nonflipped_margin_median': nonflipped.median(),
        'nonflipped_pct_decisive': 100 * (nonflipped >= 1.0).mean(),
        'mean_diff': flipped.mean() - nonflipped.mean(),
        'mw_pvalue': p,
    })

axis_x_margin = pd.DataFrame(rows)
print('Axis × verdict-margin (flipped vs non-flipped, anchor |mean_score|):')
print(axis_x_margin.round(3).to_string(index=False))

axis_x_margin.to_csv(results_dir / 'axis_x_margin.csv', index=False)
print('\nSaved: results/axis_x_margin.csv')

print('\n--- Headline reading ---')
for _, row in axis_x_margin.iterrows():
    if row.mean_diff < 0:
        story = 'flips already-borderline prompts (lower margin)'
    else:
        story = 'flips spans decisive prompts too (higher margin)'
    sig = '**' if pd.notna(row.mw_pvalue) and row.mw_pvalue < 0.05 else '  '
    p_str = f'p={row.mw_pvalue:.3f}' if pd.notna(row.mw_pvalue) else 'p=NaN'
    print(f'  {row.axis}: Δmargin={row.mean_diff:+.3f}, {row.flipped_pct_decisive:.0f}% of flipped were decisive (vs {row.nonflipped_pct_decisive:.0f}% non-flipped) — {story} ({p_str}) {sig}')

Axis × verdict-margin (flipped vs non-flipped, anchor |mean_score|):
     axis  flipped_n  flipped_margin_mean  flipped_margin_median  flipped_pct_decisive  nonflipped_n  nonflipped_margin_mean  nonflipped_margin_median  nonflipped_pct_decisive  mean_diff  mw_pvalue
        C        129                0.558                    0.5                37.209           371                   1.025                       1.0                   72.776     -0.466        0.0
        F        208                0.740                    1.0                53.846           292                   1.021                       1.0                   70.548     -0.280        0.0
        D        110                0.405                    0.0                25.455           390                   1.045                       1.0                   74.359     -0.641        0.0
       L1        173                0.549                    0.5                33.526           327                   1.093               

## 3. Axis × prompt characteristics

Surface-level features of the prompts themselves (not the responses). For each axis, check whether flipped prompts differ from non-flipped on:
- prompt length (tokens, approx via word count)
- has-code-block (contains triple backticks)
- has-numbers (contains digits — proxy for math/quantitative)

In [4]:
code_re = re.compile(r'```')
num_re = re.compile(r'\d')

def prompt_features(text):
    return pd.Series({
        'prompt_len_words': len(text.split()),
        'prompt_has_code': bool(code_re.search(text)),
        'prompt_has_numbers': bool(num_re.search(text)),
    })

prompt_feat = pairs[['uid', 'prompt']].set_index('uid')
prompt_feat[['prompt_len_words', 'prompt_has_code', 'prompt_has_numbers']] = prompt_feat['prompt'].apply(prompt_features)

rows = []
for ax, mask in axis_masks.items():
    common = prompt_feat.index.intersection(mask.index)
    sub = prompt_feat.loc[common].copy()
    sub['flipped'] = mask.loc[common]
    flipped = sub[sub.flipped]
    nonflipped = sub[~sub.flipped]
    rows.append({
        'axis': ax,
        'flipped_len_mean': flipped.prompt_len_words.mean(),
        'nonflipped_len_mean': nonflipped.prompt_len_words.mean(),
        'len_diff': flipped.prompt_len_words.mean() - nonflipped.prompt_len_words.mean(),
        'flipped_pct_code': 100 * flipped.prompt_has_code.mean(),
        'nonflipped_pct_code': 100 * nonflipped.prompt_has_code.mean(),
        'flipped_pct_numbers': 100 * flipped.prompt_has_numbers.mean(),
        'nonflipped_pct_numbers': 100 * nonflipped.prompt_has_numbers.mean(),
    })

axis_x_prompt = pd.DataFrame(rows)
print('Axis × prompt characteristics:')
print(axis_x_prompt.round(2).to_string(index=False))
axis_x_prompt.to_csv(results_dir / 'axis_x_prompt_features.csv', index=False)
print('\nSaved: results/axis_x_prompt_features.csv')

Axis × prompt characteristics:
     axis  flipped_len_mean  nonflipped_len_mean  len_diff  flipped_pct_code  nonflipped_pct_code  flipped_pct_numbers  nonflipped_pct_numbers
        C            144.75               150.50     -5.75              5.43                 2.16                85.27                   72.24
        F            140.70               154.95    -14.25              3.37                 2.74                76.92                   74.66
        D            174.99               141.69     33.30              3.64                 2.82                85.45                   72.82
       L1            143.95               151.70     -7.74              2.31                 3.36                83.82                   71.25
L1_gemini            137.29               155.85    -18.56              2.72                 3.16                80.98                   72.47

Saved: results/axis_x_prompt_features.csv


## 4. Volatility-regime characterisation

Compare six volatility regimes from per-prompt flip patterns:
- **stable_core** (176 prompts, flip_count=0)
- **maximally_unstable** (23 prompts, flip_count=4)
- **F_only** (70 prompts)
- **L1_only** (37 prompts)
- **C_only** (16 prompts)
- **D_only** (8 prompts)

In [5]:
# Build regime sets from per_prompt_volatility.csv
vol = volatility.set_index('uid')
regime_uids = {
    'stable_core': set(vol[vol.flip_count == 0].index),
    'maximally_unstable': set(vol[vol.flip_count == len(AXIS_CELLS)].index),  # [L1_GEMINI_PATCH] generalises to 4 or 5 axes
    'F_only': set(vol[vol.flipped_axes == 'F'].index),
    'L1_only': set(vol[vol.flipped_axes == 'L1'].index),
    'C_only': set(vol[vol.flipped_axes == 'C'].index),
    'D_only': set(vol[vol.flipped_axes == 'D'].index),
}
if 'L1_gemini' in AXIS_CELLS:  # [L1_GEMINI_PATCH]
    regime_uids['L1_gemini_only'] = set(vol[vol.flipped_axes == 'L1_gemini'].index)

rows = []
for regime, uids in regime_uids.items():
    common_pf = prompt_feat.index.intersection(uids)
    common_st = style_indexed.index.intersection(uids)
    common_am = anchor_margin.index.intersection(uids)
    pairs_sub = pairs.set_index('uid').loc[list(pairs.set_index('uid').index.intersection(uids))]
    rows.append({
        'regime': regime,
        'n': len(uids),
        'pct_coding': 100 * (pairs_sub['subcategory'] == 'coding').mean() if len(pairs_sub) else np.nan,
        'pct_math': 100 * (pairs_sub['subcategory'] == 'math').mean() if len(pairs_sub) else np.nan,
        'pct_non_english': 100 * (pairs_sub['language'] != 'English').mean() if len(pairs_sub) else np.nan,
        'mean_prompt_len_words': prompt_feat.loc[common_pf, 'prompt_len_words'].mean() if len(common_pf) else np.nan,
        'pct_has_code': 100 * prompt_feat.loc[common_pf, 'prompt_has_code'].mean() if len(common_pf) else np.nan,
        'mean_length_diff': style_indexed.loc[common_st, 'length_diff'].mean() if len(common_st) else np.nan,
        'mean_md_diff': style_indexed.loc[common_st, 'md_diff'].mean() if len(common_st) else np.nan,
        'mean_anchor_margin': anchor_margin.loc[common_am].mean() if len(common_am) else np.nan,
        'pct_anchor_decisive': 100 * (anchor_margin.loc[common_am] >= 1.0).mean() if len(common_am) else np.nan,
    })

regime_df = pd.DataFrame(rows)
print('Volatility-regime fingerprints (numerical):')
print(regime_df.round(2).to_string(index=False))
regime_df.to_csv(results_dir / 'regime_fingerprints.csv', index=False)
print('\nSaved: results/regime_fingerprints.csv')

Volatility-regime fingerprints (numerical):
            regime   n  pct_coding  pct_math  pct_non_english  mean_prompt_len_words  pct_has_code  mean_length_diff  mean_md_diff  mean_anchor_margin  pct_anchor_decisive
       stable_core 147       59.18     40.82            27.89                 153.99          2.04              0.11          0.02                1.33                92.52
maximally_unstable  18       50.00     50.00            50.00                 138.33         11.11              0.06          0.02                0.50                33.33
            F_only  53       45.28     54.72            22.64                 149.70          5.66              0.11          0.02                1.04                84.91
           L1_only  23       60.87     39.13            26.09                 183.61          4.35             -0.01          0.02                1.13                73.91
            C_only  13       53.85     46.15             7.69                 295.62          7.

## 5.  regex features added to fingerprint table

In [6]:
import re

# Truncation heuristic: prompt ends without sentence/code terminator.
# Honest caveat: noisy. Won't catch semantically-truncated-but-syntactically-closed
# prompts; will false-flag prompts that end in raw output (URLs, identifiers).
TERMINAL = re.compile(r'[.!?…"\'`)\]]\s*$')
CODE_FENCE_END = re.compile(r'```\s*$')

def is_truncated(text: str) -> bool:
    if not text or not text.strip():
        return True
    s = text.rstrip()
    if CODE_FENCE_END.search(s):
        return False
    return not bool(TERMINAL.search(s))

# Multi-part: enumerated sub-questions only (skip ?-counting; too noisy on
# rhetorical/clarifying questions). Catches "1.", "1)", "(1)", "Q1", "Part 1".
SUBQ = re.compile(
    r'(?:^|\n)\s*(?:\d+[.)]|\(\d+\)|Q\d+|Part\s*\d+)',
    re.IGNORECASE,
)

def n_subquestions(text: str) -> int:
    return len(SUBQ.findall(text)) if text else 0

pairs['is_truncated']    = pairs['prompt'].apply(is_truncated)
pairs['n_subquestions']  = pairs['prompt'].apply(n_subquestions)
pairs['is_multipart']    = pairs['n_subquestions'] >= 2

# Re-tabulate per regime
def regime_features(uids):
    sub = pairs[pairs.uid.isin(uids)]
    return {
        'pct_truncated':       100 * sub.is_truncated.mean(),
        'pct_multipart':       100 * sub.is_multipart.mean(),
        'mean_n_subquestions': sub.n_subquestions.mean(),
    }

extra = pd.DataFrame(
    {r: regime_features(uids) for r, uids in regime_uids.items()}
).T
extra.index.name = 'regime'

# Merge into existing fingerprint CSV
fp_path = results_dir / 'regime_fingerprints.csv'
fp = pd.read_csv(fp_path)
fp = fp.merge(extra, left_on='regime', right_index=True, how='left')
fp.to_csv(fp_path, index=False)

print(fp[['regime', 'n',
          'pct_truncated', 'pct_multipart', 'mean_n_subquestions']]
      .round(2).to_string(index=False))


            regime   n  pct_truncated  pct_multipart  mean_n_subquestions
       stable_core 147          50.34           4.76                 0.25
maximally_unstable  18          55.56           5.56                 0.44
            F_only  53          43.40           5.66                 1.13
           L1_only  23          47.83           8.70                 0.43
            C_only  13          38.46           7.69                 0.15
            D_only   4           0.00           0.00                 0.00
    L1_gemini_only  26          57.69          11.54                 0.46


## 6. Hand-picked example prompts

Print 5 prompts per regime as raw material for hand-selecting 2–3 representative examples per regime for §5/appendix. Replaces the dropped LLM-tagging approach (cells 11, 13 in earlier draft) — more honest, fully reproducible, no LLM in the loop.

Seed=42 so the sample is stable across re-runs.

In [7]:
import textwrap

rng = np.random.default_rng(42)
N_SHOW = 5  # show 5 per regime; pick 2-3 by hand

for regime, uids in regime_uids.items():
    uid_list = list(uids)
    sample = rng.choice(uid_list, size=min(N_SHOW, len(uid_list)), replace=False)
    print(f'\n{"="*70}')
    print(f'{regime} (n_total={len(uid_list)}, showing {len(sample)})')
    print(f'{"="*70}')
    for uid in sample:
        row = pairs.loc[pairs.uid == uid].iloc[0]
        snippet = textwrap.shorten(row['prompt'], width=500, placeholder=' …')
        is_trunc = '⚠ truncated' if row.get('is_truncated', False) else ''
        n_sub = row.get('n_subquestions', 0)
        sub_tag = f'[{n_sub} sub-qs]' if n_sub >= 2 else ''
        print(f'\n[{uid}] {is_trunc} {sub_tag}'.strip())
        print(snippet)


stable_core (n_total=147, showing 5)
[5e6119f356de468a] ⚠ truncated [3 sub-qs]
Deskripsi Diberikan sebuah grid yang tersusun oleh N N baris dan M M kolom. Baris-barisnya dinomori dari 1 1 hingga N N dari atas ke bawah. Kolom-kolomnya dinomori dari 1 1 hingga M M dari kiri ke kanan. Awal mulanya semua petak berwarna putih. Terdapat barisan C C dan A A yang masing-masing terdiri dari M M bilangan. Tidak hanya itu, terdapat juga barisan R R yang terdiri dari N N bilangan. Anda, sebagai asisten Pak Chanek, diminta untuk mewarnai seluruh petak pada grid tersebut menjadi merah …
[84a773f43c2a44cf]
4 + 3 = 21 ，2 + 5 = 35， 7 + 4 = ??
[06add81598044afd] ⚠ truncated
write me a python script to download music from yt without api
[2700a551d7854f1c]
find the last non-zero digit of 50!
[6cab48567fb845e6] ⚠ truncated
write a simple todo cmd app in python with a option to see the current todos but each incompleted todo is marked with a natural number and each completed todo is marked with a prime num

In [8]:
# Section 7: Headline summary table for §5
summary_rows = []

# Per-axis fingerprints (cells 1, 2, 3)
for ax in AXIS_CELLS:
    margin_row = axis_x_margin[axis_x_margin.axis == ax].iloc[0]
    prompt_row = axis_x_prompt[axis_x_prompt.axis == ax].iloc[0]
    summary_rows.append({
        'level': 'axis',
        'name': ax,
        'n_flipped': int(axis_masks[ax].sum()),
        'flipped_margin_mean': round(margin_row.flipped_margin_mean, 3),
        'flipped_pct_decisive': round(margin_row.flipped_pct_decisive, 1),
        'flipped_len_mean': round(prompt_row.flipped_len_mean, 1),
        'flipped_pct_code': round(prompt_row.flipped_pct_code, 1),
        'fingerprint': {
            'C': 'long, code-heavy, number-rich',
            'F': 'short, anchor-decisive overridable',
            'D': 'long, number-heavy',
            'L1': 'prose-heavy, half the code rate of base',
            'L1_gemini': 'prose-heavy (Gemini judge), parallel L1 axis',  # [L1_GEMINI_PATCH]
        }[ax],
    })

# Per-regime fingerprints (cell 9 + 9b)
for _, r in fp.iterrows():
    summary_rows.append({
        'level': 'regime',
        'name': r.regime,
        'n_flipped': int(r.n),
        'flipped_margin_mean': round(r.mean_anchor_margin, 3),
        'flipped_pct_decisive': round(r.pct_anchor_decisive, 1),
        'flipped_len_mean': round(r.mean_prompt_len_words, 1),
        'flipped_pct_code': round(r.pct_has_code, 1),
        'fingerprint': {
            'stable_core': 'anchor-decisive, no axis flips',
            'maximally_unstable': 'non-English-heavy, anchor-borderline',
            'F_only': 'short, mostly English, anchor-decisive',
            'C_only': 'long, code-heavy, English-dominant',
            'L1_only': 'coding-heavy',
            'D_only': 'math-heavy, longer responses',
        }.get(r.regime, ''),
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

 level               name  n_flipped  flipped_margin_mean  flipped_pct_decisive  flipped_len_mean  flipped_pct_code                                  fingerprint
  axis                  C        129                0.558                  37.2             144.8               5.4                long, code-heavy, number-rich
  axis                  F        208                0.740                  53.8             140.7               3.4           short, anchor-decisive overridable
  axis                  D        110                0.405                  25.5             175.0               3.6                           long, number-heavy
  axis                 L1        173                0.549                  33.5             144.0               2.3      prose-heavy, half the code rate of base
  axis          L1_gemini        184                0.595                  41.8             137.3               2.7 prose-heavy (Gemini judge), parallel L1 axis
regime        stable_core        1